# Classificador FakeClientID SOME/IP — Pipeline Completo

Notebook de reprodução completa: do PCAP bruto até as métricas finais.

**Ataque**: FakeClientID — rotação de `client_id` em mensagens REQUEST  
**Dataset**: Alkhatib et al. (2021). Anomaly Detection Dataset for SOME/IP Protocol.  
**Diferença**: labels pré-definidos em CSV (não derivados por conteúdo SOME/IP como no Kim).

In [ ]:
import sys, time
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
)

FCI_DIR    = Path().resolve()          # detection/fake_client_id/
DET_DIR    = FCI_DIR.parent            # detection/
ROOT_DIR   = DET_DIR.parent            # SDV_Research/
RAW_DIR    = DET_DIR / 'data' / 'raw'
FRAMES_DIR = ROOT_DIR / 'experiments' / 'someip_traces' / 'dataframes'
DATA_DIR   = FCI_DIR / 'data'
MODEL_DIR  = FCI_DIR / 'model'

FEAT_COLS = [
    'f01_ip_time_interval', 'f08_someip_payload_change',
    'f11_ip_length_change', 'f12_tcpudp_length_change',
    'f13_payload_repeat_rate', 'f15_someip_payload_len',
    'f16_tcpudp_len', 'f17_src_packet_rate',
    'f18_src_payload_diversity', 'f19_is_sd',
    'f20_src_service_diversity', 'f21_is_relay_service',
    'f22_src_clientid_diversity',
]
FEAT_LABELS = [
    'f01 ip_time_interval', 'f08 payload_hamming',
    'f11 ip_len_change', 'f12 tcpudp_len_change',
    'f13 payload_repeat', 'f15 someip_payload_len',
    'f16 tcpudp_len', 'f17 src_packet_rate',
    'f18 payload_diversity', 'f19 is_sd',
    'f20 service_diversity', 'f21 is_relay',
    'f22 clientid_diversity',
]

PCAP_SOURCES = [
    ('fakeClientID.pcap',  'fakeclientid1.csv'),
    ('fakeClientID2.pcap', 'fakeclientid2.csv'),
]

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Ambiente configurado.')
print(f'  fake_client_id/ : {FCI_DIR}')
print(f'  dataframes/     : {FRAMES_DIR}')

## 1. Dataset e Cenário de Ataque

### 1.1 Visão geral

O dataset vem do trabalho **Alkhatib et al. (2021)**, que disponibiliza PCAPs de ataques SOME/IP
junto com arquivos CSV contendo o label de cada pacote já definido pelos autores.
Não é necessário derivar os labels — eles são fornecidos diretamente.

### 1.2 O ataque FakeClientID

O campo `client_id` (2 bytes no offset 8 do cabeçalho SOME/IP) identifica unicamente
o cliente em uma sessão de requisição-resposta. Em operação normal, cada ECU usa um
`client_id` fixo por sessão — o servidor usa esse campo para rotear respostas.

O atacante **rotaciona múltiplos `client_ids` falsos** em sequência nas mensagens REQUEST.
Isso provoca:
- Confusão na tabela de sessões do servidor (associações inválidas IP↔client_id)
- Potencial esgotamento de recursos (CPU/memória) por criação de entradas fantasma
- Dificuldade de rastreamento: não há um único `client_id` ligado ao atacante

### 1.3 PCAPs utilizados

| PCAP | CSV de labels | Observação |
|---|---|---|
| `fakeClientID.pcap` | `fakeclientid1.csv` | Primeira captura |
| `fakeClientID2.pcap` | `fakeclientid2.csv` | Segunda captura |

In [ ]:
print('=== Arquivos de entrada ===')
for pcap_name, csv_name in PCAP_SOURCES:
    pcap_path = RAW_DIR / pcap_name
    csv_path  = FRAMES_DIR / csv_name
    for p, label in [(pcap_path, 'PCAP'), (csv_path, 'CSV')]:
        status = f'{p.stat().st_size/1e6:.1f} MB  [OK]' if p.exists() else 'NAO ENCONTRADO'
        print(f'  {label}: {p.name:<30}  {status}')
    print()

## 2. Extração de Features

**13 features** extraídas por pacote com janelas deslizantes stateful.
A feature nova `f22_src_clientid_diversity` conta o número de `client_ids` distintos
emitidos pelo mesmo `src_ip` em uma janela de 100 REQUESTs consecutivos.

| Feature | Descrição |
|---|---|
| f01 | ip_time_interval — intervalo temporal entre pacotes do fluxo |
| f08 | someip_payload_change — Hamming entre payloads consecutivos |
| f11 | ip_length_change — variação de tamanho IP |
| f12 | tcpudp_length_change — variação de tamanho TCP/UDP |
| f13 | payload_repeat_rate — fração dos últimos 5 payloads idênticos |
| f15 | someip_payload_len — tamanho do payload SOME/IP |
| f16 | tcpudp_len — comprimento total TCP/UDP |
| f17 | src_packet_rate — taxa de pacotes por IP (janela 1000) |
| f18 | src_payload_diversity — payloads únicos / total (janela 1000) |
| f19 | is_sd — flag SOME/IP-SD (svc=0xFFFF) |
| f20 | src_service_diversity — service_ids únicos do IP (janela 100) |
| f21 | is_relay_service — flag relay service (svc=0x100B) |
| **f22** | **src_clientid_diversity — client_ids únicos do IP em REQUESTs (janela 100)** |

**Normalização**: Min-Max [0,1] calculada no total, sem data leakage entre treino e teste.  
**Split**: 70/30 estratificado por label, `random_state=42`.

In [ ]:
X_train = np.load(DATA_DIR / 'X_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy').astype(int)
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy').astype(int)

# Remap: 5 -> 1 para XGBoost binario
y_train_bin = (y_train == 5).astype(int)
y_test_bin  = (y_test  == 5).astype(int)

n_total    = len(y_train) + len(y_test)
n_ben_all  = (y_train_bin == 0).sum() + (y_test_bin == 0).sum()
n_fci_all  = (y_train_bin == 1).sum() + (y_test_bin == 1).sum()

split_data = pd.DataFrame({
    'Conjunto': ['Treino (70%)', 'Teste (30%)', 'Total'],
    'Amostras': [len(y_train), len(y_test), n_total],
    'Benigno':  [(y_train_bin==0).sum(), (y_test_bin==0).sum(), n_ben_all],
    'FakeClientID': [(y_train_bin==1).sum(), (y_test_bin==1).sum(), n_fci_all],
    '% Ataque': [
        f'{100*(y_train_bin==1).sum()/len(y_train):.1f}%',
        f'{100*(y_test_bin==1).sum()/len(y_test):.1f}%',
        f'{100*n_fci_all/n_total:.1f}%',
    ],
})
display(split_data.style.hide(axis='index'))
print(f'\nFeatures: {X_train.shape[1]}  |  Shape treino: {X_train.shape}  |  Shape teste: {X_test.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].bar(['Benigno', 'FakeClientID'], [n_ben_all, n_fci_all], color=['#4c9be8', '#e05c5c'])
axes[0].bar_label(axes[0].containers[0],
                  labels=[f'{n_ben_all:,}\n({100*n_ben_all/n_total:.1f}%)',
                          f'{n_fci_all:,}\n({100*n_fci_all/n_total:.1f}%)'],
                  padding=4, fontsize=9)
axes[0].set_title('Distribuicao total do dataset')
axes[0].set_ylabel('Pacotes')
axes[0].set_ylim(0, max(n_ben_all, n_fci_all) * 1.25)

x = np.arange(2); w = 0.35
b_tr = (y_train_bin==0).sum(); a_tr = (y_train_bin==1).sum()
b_te = (y_test_bin==0).sum();  a_te = (y_test_bin==1).sum()
axes[1].bar(x - w/2, [b_tr, b_te], w, label='Benigno',      color='#4c9be8')
axes[1].bar(x + w/2, [a_tr, a_te], w, label='FakeClientID', color='#e05c5c')
axes[1].set_xticks(x); axes[1].set_xticklabels(['Treino (70%)', 'Teste (30%)'])
axes[1].set_title('Split Treino / Teste')
axes[1].set_ylabel('Pacotes')
axes[1].legend()

plt.tight_layout()
plt.savefig(FCI_DIR / 'dataset_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Treinamento

**Algoritmo**: XGBoost binário (benigno=0 vs FakeClientID=1).  
**Parâmetros**: `n_estimators=200`, `max_depth=6`, `learning_rate=0.1`, `subsample=0.8`,
`colsample_bytree=0.8`, `scale_pos_weight=3.2` (ajuste pelo desbalanceamento 3.2:1).

In [ ]:
scale_pos = int((y_train_bin == 0).sum()) / max(int((y_train_bin == 1).sum()), 1)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    tree_method='hist',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

t0 = time.perf_counter()
model.fit(X_train, y_train_bin, verbose=False)
t_train = time.perf_counter() - t0

single = X_test[:1]
_ = model.predict(single)
t0 = time.perf_counter()
for _ in range(1000):
    model.predict(single)
t_single_ms = (time.perf_counter() - t0) / 1000 * 1000

t0 = time.perf_counter()
y_pred = model.predict(X_test)
t_batch = time.perf_counter() - t0
y_prob  = model.predict_proba(X_test)[:, 1]

MODEL_DIR.mkdir(exist_ok=True)
model.save_model(str(MODEL_DIR / 'fakeclientid_classifier.json'))

print(f'Tempo de treinamento : {t_train:.3f} s')
print(f'Inferencia batch     : {t_batch:.4f} s  ({len(X_test):,} amostras)')
print(f'Latencia por pacote  : {t_single_ms:.3f} ms')
print(f'Throughput           : {len(X_test)/t_batch:,.0f} pkt/s')
print(f'Modelo salvo em      : {MODEL_DIR}/fakeclientid_classifier.json')

## 4. Métricas de Classificação

In [ ]:
acc  = accuracy_score(y_test_bin, y_pred)
prec = precision_score(y_test_bin, y_pred, zero_division=0)
rec  = recall_score(y_test_bin, y_pred, zero_division=0)
f1   = f1_score(y_test_bin, y_pred, zero_division=0)
auc  = roc_auc_score(y_test_bin, y_prob)
cm   = confusion_matrix(y_test_bin, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr_val = fn / (fn + tp) if (fn + tp) > 0 else 0

summary = pd.DataFrame({
    'Metrica': [
        'Acuracia', 'Precision', 'Recall (TPR)', 'F1-Score', 'AUC-ROC',
        'Verdadeiros Negativos (TN)', 'Falsos Positivos (FP)',
        'Falsos Negativos (FN)', 'Verdadeiros Positivos (TP)',
        'Taxa Falso Positivo (FPR)', 'Taxa Falso Negativo (FNR)',
    ],
    'Valor': [
        f'{acc:.4f}', f'{prec:.4f}', f'{rec:.4f}', f'{f1:.4f}', f'{auc:.6f}',
        f'{tn:,}', f'{fp:,}', f'{fn:,}', f'{tp:,}',
        f'{fpr_val:.6f}', f'{fnr_val:.6f}',
    ]
})
display(summary.style.hide(axis='index'))

## 5. Visualizações

In [ ]:
fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# Matriz de confusão
ax0 = fig.add_subplot(gs[0])
disp = ConfusionMatrixDisplay(cm, display_labels=['Benigno', 'FakeClientID'])
disp.plot(ax=ax0, colorbar=False, cmap='Blues')
ax0.set_title('Matriz de Confusao')

# Curva ROC
ax1 = fig.add_subplot(gs[1])
fpr_c, tpr_c, _ = roc_curve(y_test_bin, y_prob)
ax1.plot(fpr_c, tpr_c, lw=2, color='#9b59b6', label=f'AUC = {auc:.4f}')
ax1.fill_between(fpr_c, tpr_c, alpha=0.08, color='#9b59b6')
ax1.plot([0,1],[0,1],'--', color='grey', lw=1)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('Curva ROC')
ax1.legend(loc='lower right')
ax1.set_xlim(-0.01, 1.01); ax1.set_ylim(-0.01, 1.01)

# Importância das features
ax2 = fig.add_subplot(gs[2])
imps  = model.feature_importances_
order = np.argsort(imps)
colors = ['#e05c5c' if imps[i] > 0.05 else '#4c9be8' for i in order]
bars = ax2.barh([FEAT_LABELS[i] for i in order], imps[order], color=colors)
ax2.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
ax2.set_xlabel('Importancia (gain)')
ax2.set_title('Importancia das Features')
ax2.set_xlim(0, max(imps) * 1.18)

plt.suptitle('Classificador FakeClientID SOME/IP — XGBoost (13 features)', fontsize=12, y=1.02)
plt.savefig(FCI_DIR / 'results_overview.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Discussão

### Desempenho
O classificador atinge **F1=0,9982** com 13 features comportamentais, sem acesso ao conteúdo dos bytes.
Duas features explicam 96,6% da importância:

- **f20 src_service_diversity (67,5%)**: o atacante, ao rotacionar client_ids em requisições a
  múltiplos serviços, usa mais service_ids do que ECUs legítimas especializadas.
  Uma ECU real normalmente oferece/requisita 1-2 serviços fixos.

- **f22 src_clientid_diversity (29,1%)**: a feature projetada especificamente para este ataque.
  Conta client_ids únicos em REQUESTs nos últimos 100 pacotes do IP. O atacante rotaciona
  dezenas de client_ids; ECUs legítimas usam 1.

### Diferença em relação aos outros classificadores

| Aspecto | Kim (DoS/Fuzzy/MITM) | Alkhatib (FakeClientID) |
|---|---|---|
| Origem dos labels | Derivado da spec SOME/IP | Fornecido pelos autores em CSV |
| Tamanho do dataset | ~2–4 M amostras | 3.920 amostras |
| Split | 50/50 | 70/30 |
| Feature extra | — | f22 src_clientid_diversity |

### Erros residuais
- **Falsos Positivos**: zero — nenhuma ECU legítima atingiu o padrão de diversidade de client_ids.
- **Falso Negativo (1)**: provavelmente o primeiro pacote do atacante, antes da janela de
  f22 acumular client_ids suficientes para ultrapassar o threshold do modelo.

### Limitações
| Cenário | Detectado? | Motivo |
|---|---|---|
| Rotação rápida de client_ids (este dataset) | Sim | f20+f22 discriminam perfeitamente |
| ECU legítima com muitos serviços | Possível FP | f20 pode subir acima do threshold |
| FakeClientID com client_ids próximos | Não detectado | f22 ficaria baixo |
| Dataset pequeno (~4k) | Limitação | Generalização não testada em capturas maiores |